# Introduction

`pyiron_workflow` is a workflow management system (WfMS) built on top of `flowrep` workflow schemas. It has two key roles:

1. Transform `flowrep` prospective class-view recipe objects into `flowrep` retrospective instance-view data objects by running them with a given set of input data. This process should be empowered to...
  - Associate additional runtime metadata to `flowrep` results
  - Caching results for
  - Apply computational resources to individual nodes in the graph (i.e. executors)
  - Offer hooks for tracking execution status and failures
  - Give access to recipe validation tools beyond basic `flowrep` recipe structure validation -- i.e. python type hints, and ontological validation via `semantikon`
2. Make it easier to write `flowrep` recipes be...
  - Providing a mutable object which allows _transiently_ invalid recipes during the construction process.
  - Giving quality-of-life (QoL) support in the form of easy hooks and handles so both users and other software (e.g. a GUI) can manage, undo, redo, and fail gracefully from graph recipe mutations.



# Running `flowrep` recipes

`flowrep` turns ordinary, decorated python functions into _recipes_: declarative, data-only descriptions of a computation. Recipes are not runnable on their own --
they hold no input values and no results.
`pyiron_workflow` is the runtime that uses the recipe to generate a `flowrep` retrospective data object, and fill it up according to initial input data and the recipe.

We could open up our favourite text editor and write a `flowrep` recipe by hand, but it's easier to parse the recipe from decorated python.
For a complete introduction to `flowrep`, take a look [at its user guide](https://github.com/pyiron/flowrep/blob/main/notebooks/user-guide.ipynb).
Here, just recall that "atomic" nodes have no internal graph structure, and you can typically even omit decorating them, while workflow _must_ have a decorator and are written as lightly-constrained but natural-feeling python.
Workflows can be nested and handle standard flow controlls of for-each (i.e. parallel maps), while, if/else, and try/except.

In [1]:
import flowrep as fr

# @fr.atomic  # Nice to signal intent, but usually not necessary
def add(x, y):
    return x + y

@fr.atomic
def mul(x, y):
    return x * y

@fr.workflow
def line(x, m, b):
    scaled = mul(x, m)
    shifted = add(scaled, b)
    return shifted

Decorating attaches a `flowrep_recipe` to the function. This is the declarative recipe (which is a `pydantic` model):

In [2]:
mul.flowrep_recipe.model_dump(mode="json")

{'type': 'atomic',
 'inputs': ['x', 'y'],
 'outputs': ['output_0'],
 'description': None,
 'reference': {'info': {'module': '__main__',
   'qualname': 'mul',
   'version': None},
  'inputs_with_defaults': [],
  'restricted_input_kinds': {}}}

## The `node` entry point

`pwf.node` is the most generic way into the `pyiron_workflow` runtime: hand it a recipe (or a decorated function, or an existing node, or even an undecorated function to be treated as an atomic node!) and it returns a live `Node`.
Atomic recipes become `Atomic` nodes; workflow recipes become `Macro` nodes.

A node exposes named input and output _ports_, the `recipe` it came from, and its `label`.
(A node also carries an `executor` and a `last_run`; we meet those further down, once they have something to do.)

In [3]:
import pyiron_workflow._wfms.api as pwf

In [4]:
n = pwf.node(add)
print("type:    ", type(n).__name__)
print("inputs:  ", list(n.inputs.keys()))
print("outputs: ", list(n.outputs.keys()))
print("label:   ", n.label)
n.recipe

type:     Atomic
inputs:   ['x', 'y']
outputs:  ['output_0']
label:    add


AtomicRecipe(type=<RecipeElementType.ATOMIC: 'atomic'>, inputs=['x', 'y'], outputs=['output_0'], description=None, reference=PythonReference(info=VersionInfo(module='__main__', qualname='add', version=None), inputs_with_defaults=[], restricted_input_kinds={}))

## Running a node and reading its provenance

A workflow recipe becomes a `Macro` -- a node that owns child nodes and the edges between them.
Calling `.run(**input_data)` executes it and returns a `Run` object.

The `Run` is _retrospective provenance_: it is a record of what actually happened, holding input and output data for each constituent node as their raw python objects.
It carries a `flowrep` data `result`, the `recipe` that generated it, a `status`, timing (`duration`), a shortcut to the terminal `outputs`, and a `steps` tree recording the sub-`Run` of every child that executed (which is linked to the graph-structured `result`).


In [5]:
m = pwf.node(line)
run = m.run(x=2, m=5, b=1)

In [6]:
print("status:  ", run.status)
print("outputs: ", {k: v.value for k, v in run.outputs.items()})
print("duration:", run.duration)
print("steps:   ", run.steps.labels)

status:   finished
outputs:  {'shifted': 11}
duration: 0:00:00.000576
steps:    ['mul_0', 'add_0']


You can drill into each child's own status and outputs via the `result` or `steps` -- these are different access paths to the same reference-linked data:

In [7]:
for step in run.steps:
    print(step.label, step.status, {k: v for k, v in step.outputs.items()})

mul_0 finished {'output_0': OutputDataPort(value=10, annotation=None)}
add_0 finished {'output_0': OutputDataPort(value=11, annotation=None)}


In [8]:
for child_label, result_subnode in run.result.nodes.items():
    for k, v in result_subnode.output_ports.items():
        print(child_label, {k: v.value})

mul_0 {'output_0': 10}
add_0 {'output_0': 11}


## The `@pwf.atomic` / `@pwf.workflow` decorators

Writing `pwf.node(fn)` every time works, but the `pyiron_workflow` decorators bake the runtime conveniences right onto the decorated function via a `.pwf` handle: `fn.pwf.node()` builds a node, `fn.pwf.run(...)` builds-and-runs in one step, and (for workflows) `fn.pwf.validate()` checks the graph.
These are pure extensions of the `flowrep` decorators -- they leave the decorated functions as plain functions, leave the `.flowrep_recipe` in place, and only _add_ the `.pwf` helper.

In [9]:
@pwf.atomic
def increment(x):
    return x + 1

In [10]:
@pwf.workflow
def affine(x, m):
    scaled = mul(x, m)
    bumped = increment(scaled)
    return bumped

In [11]:
print("one-shot run:", increment.pwf.run(x=4).outputs["output_0"])

one-shot run: OutputDataPort(value=5, annotation=None)


In [12]:
node = affine.pwf.node()
node.run(x=2, m=5)
# After running, the node remembers its most recent Run on `.last_run`:
print("last_run:", node.last_run.status, node.last_run.outputs["bumped"].value)

last_run: finished 11


### Pull-mode

_(TO IMPLEMENT)_ -- running a single output port and pulling only the upstream
nodes it depends on. Coming soon.

## RunConfig

A `RunConfig` carries run settings like where progress artefacts live, lists of hooks, and other utilities.
One gets generated by default on each run invocation, but you can also fine-tune behaviour by passing one in explicitly.
Internally, the `RunConfig` will track the lexical path of the `.prime_mover` of the run, i.e. the node with whom `.run` was called.

### Status hooks

`progress_hooks` fire on every status transition of every node in the tree,
receiving the progress directory, a timestamp, the node's lexical path, and the new
status. This is the seam a GUI or logger taps to follow execution live.
Each hook is wrapped in a `ProgressHook(fn, blocking=False)`.
By default hooks are **non-blocking** — they run on a background thread pool (sized by `RunConfig.hooks_max_threads`) so a slow hook can't stall execution, and a hook that raises is logged (to `RunConfig.logger_name`) rather than failing the run.
Pass `blocking=True` for a hook that must run inline and whose failure should abort the run.

In [13]:
events = []

def progress_hook(run_dir, time, lexical_path, status):
    events.append((lexical_path, str(status)))

wf = pwf.node(line)
cfg = pwf.RunConfig(progress_hooks=[pwf.ProgressHook(progress_hook, blocking=True)])
wf.run(cfg, x=1, m=2, b=3)

for lexical_path, status in events:
    print(f"{lexical_path} {status}")

line running
line.mul_0 running
line.mul_0 finished
line.add_0 running
line.add_0 finished
line finished


### Failure hooks

`exception_hooks` fire when the prime mover fails, receiving the failed `Run` so you can inspect (and persist) the exact state at the moment of failure. This gives you the opportunity to serialize it -- here with `pickle` -- and reload it later to perform a post-mortem without re-running anything.

In [14]:
import pickle

@pwf.atomic
def reciprocal(x):
    return 1 / x  # raises ZeroDivisionError when x == 0

@pwf.atomic
def decrement(x):
    return x - 1

@pwf.atomic
def positive(x):
    return x > 0

@pwf.workflow
def decrement_while_positive(x, r=0):
    while positive(x):
        x = decrement(x)
        r = reciprocal(x)
    return r

decrement_while_positive.pwf.run(x=3.1).outputs, (1 / (3.1 - 4))

({'r': OutputDataPort(value=-1.1111111111111112, annotation=None)},
 -1.1111111111111112)

Now that we know our workflow is working,

In [15]:
captured = {}

def exception_hook(run_dir, failed_run, exception):
    captured["run"] = failed_run

wf = pwf.node(decrement_while_positive, "doomed")

cfg = pwf.RunConfig(exception_hooks=[exception_hook])

try:
    wf.run(cfg, x=3)
except ZeroDivisionError:
    print("the run raised, as expected")

the run raised, as expected


In [16]:
# Round-trip the captured failure state through pickle and inspect it:
reloaded = pickle.loads(pickle.dumps(captured["run"]))
print("reloaded status:   ", reloaded.status)
print("reloaded exception:", repr(reloaded.exception))

reloaded status:    failed
reloaded exception: ZeroDivisionError('division by zero')


We can dig into the results leading up to the failure

In [17]:
for k, subresult in reloaded.result.nodes["while_0"].nodes.items():
    if k.startswith("body_"):
        print(f"{k} outputs:", subresult.output_ports)

body_0 outputs: {'x': OutputDataPort(value=2, annotation=None), 'r': OutputDataPort(value=0.5, annotation=None)}
body_1 outputs: {'x': OutputDataPort(value=1, annotation=None), 'r': OutputDataPort(value=1.0, annotation=None)}
body_2 outputs: {'x': OutputDataPort(value=NOT_DATA, annotation=None), 'r': OutputDataPort(value=NOT_DATA, annotation=None)}


Right down to the outputs of the last decrementing step

In [18]:
while_loop = reloaded.steps[0]
last_body = while_loop.steps[-1]
last_decrement = last_body.steps[0]
last_decrement.outputs

{'output_0': OutputDataPort(value=0, annotation=None)}

And the inputs of that triggered the failure

In [19]:
last_reciprocal = last_body.steps[1]
last_reciprocal.result.input_ports

{'x': InputDataPort(value=0, annotation=None, default=NOT_DATA)}

For real archival you will want something sturdier and more powerful than `pickle`
-- [`bagofholding`](https://github.com/pyiron/bagofholding/) is a good fit for durably storing these result objects.

## Executors

`pyiron_workflow` uses the `concurrent.futures.Executor` to parallelize operations.
By default, all the layers in a DAG get multithreaded with `concurrent.futures.ThreadPoolExecutor`:



In [20]:
import datetime
import time

def sleepy(t_sleep):
    time.sleep(t_sleep)
    return None

@pwf.workflow
def sleep_array(times: list[float | int]):
    slept_for = []
    for t in times:
        _ = sleepy(t)
        slept_for.append(t)
    return slept_for

macro = sleep_array.pwf.node()

start = datetime.datetime.now()
macro.run(times=[0.1, 0.2, 0.3, 0.4])
elapsed = datetime.datetime.now() - start
print(elapsed.total_seconds())

0.407783


We see here that the runtime is approximately equal to the largest sleeping time, not their sum.

We can use the `pwf.RunConfig` object discussed in the last section to control this parallelism, including the max threads, whether child errors are grouped and raised together or greedily raised as soon as they're encountered, or to turn off threading entirely.
If we do that last, we can see the runtime balloons to the sum of all sleep times:

In [21]:
cfg = pwf.RunConfig(dag_layers_multithreaded=False)

start = datetime.datetime.now()
macro.run(
    cfg,
    times=[0.1, 0.2, 0.3, 0.4],
)
elapsed = datetime.datetime.now() - start
print(elapsed.total_seconds())

1.016259


We can also control the execution of an individual node by explicitly setting its `.executor` field, which accepts a `concurrent.futures.Executor` instance, or `pwf.ExecutorInstructions` on how to build one.
This "instructions" format is critical for nesting out-of-process execution, which requires the outer executor to serialize the inner executor;
Since executors are not straightforwardly serializable, we thus require any inner executors to be by-instruction.

Typical `concurrent.futures.ProcessPoolExecutor` out-of-process execution also requires that the underlying functions be pickleable and importable.
That importability requirement rules out using anything we define here in the notebook, so we'll pull a function from python's standard library.

However, this introduces a new hiccup: many of the stdlib functions are not trivially parseable.
Thankfully, `flowrep` gives us the flexibility to make a runable recipe out of anything with a signature:

In [22]:
import os

getpid_recipe = fr.parse_atomic(
    os.getpid,
    "pid",
)

We won't go as far as nesting executors here, but we'll still demonstrate the instructions based syntax:

In [23]:
from concurrent import futures

@fr.workflow
def pids():
    pid0 = getpid_recipe()
    pid1 = getpid_recipe()
    return pid0, pid1

wf = pwf.node(pids)
wf.getpid_1.executor = pwf.ExecutorInstructions(
    futures.ProcessPoolExecutor,  # Class of executor to instantiate
    # args tuple, or nothing
    # kwargs dict, or nothings
)
out = wf.run()

print("Local PID", os.getpid())
for k, v in out.outputs.items():
    print(f"{k} PID: {v.value}")

Local PID 75208
pid0 PID: 75208
pid1 PID: 75216


### Executorlib -- scaling and trans-shutdown persistence

[`executorlib`](https://github.com/pyiron/executorlib) is the pyiron solution for scaling workflows to HPC; it uses the `concurrent.futures.Executor` API, but executes pyiron functions on more powerful compute using SLURM and flux.

Because they follow the python standard library interface, `executorlib` executors are drop-in compatible in the `.executor` attribute on nodes (in both their instance and `ExecutorInstructions` format. To see that, let's revisit our process ID example using `executorlib.SingleNodeExecutor`. This time, we'll leverage `executorlib`'s use of `cloudpickle` to show how the `SingleNodeExecutor` will let us ship functions _defined locally in the notebook_ off to alternate processes.

In [24]:
import executorlib

def local_getpid():
    pid = os.getpid()
    return pid

@pwf.workflow
def local_pids():
    pid0 = local_getpid()
    pid1 = local_getpid()
    return pid0, pid1

wf = local_pids.pwf.node()
with executorlib.SingleNodeExecutor() as exe:
    wf.local_getpid_1.executor = exe
    out = wf.run()

print("Notebook process PID:", os.getpid())
out.outputs

Notebook process PID: 75208


{'pid0': OutputDataPort(value=75208, annotation=None),
 'pid1': OutputDataPort(value=75218, annotation=None)}

To handle cross-process executions (e.g. you submit a job, shutdown your jupyter notebook, then come back tomorrow and want to recover the same job), `executorlib` uses local caches. In order to get cache hits in the graph-node-execution context, we provide thinly-wrapped executors which _only_ work with submitting node runs, and which override the caching directory to use the run configuration directory, and override the cache filenames to use the lexical path.

Since we don't have access to SLURM from inside this notebook, we'll demonstrate the behaviour with a test executor. The `pyiron_workflow` wrapping aspect is identical for both this toy executor and its full SLURM counterpart, only the actual SLURM management itself differs. So here we learn cache management, and to learn how to configure the SLURM executor, see the `executorlib` docs themselves, or the GitHub continuous integration (CI) workflows for this package.

In [25]:
import pathlib
import shutil

def slow_node(t):
    time.sleep(t)
    return t

@pwf.workflow
def slow_wf(t_sleep):
    slept_for = slow_node(t_sleep)
    return slept_for

wf = slow_wf.pwf.node()
run_dir = pathlib.Path("./testing_exec_cache")
cfg = pwf.RunConfig(run_dir=run_dir)
wf.slow_node_0.executor = pwf.ExecutorInstructions(
    pwf.tools._CacheTestExecutor,
)

t_sleep = 1
t0 = time.perf_counter()
out1 = wf.run(cfg, t_sleep=t_sleep)
t1 = time.perf_counter()
out2 = wf.run(cfg, t_sleep=t_sleep)
t2 = time.perf_counter()

print(f"First run took {t1-t0:.3f}s")
print(f"Second run took {t2-t1:.3f}s")
cache_dir = run_dir / pwf.tools._CacheTestExecutor.cache_directory
print("Second run exploited executorlib cache:", list(cache_dir.iterdir()))

First run took 2.673s
Second run took 0.028s
Second run exploited executorlib cache: [PosixPath('testing_exec_cache/executorlib_cache/pyiron.log'), PosixPath('testing_exec_cache/executorlib_cache/slow_wf.slow_node_0_o.h5')]


Since the cache filename derives from the node's lexical path, _all_ invocations of this executor for the same node in the same run config directory will find the hash -- even if we don't want them to. E.g., below we change our input, but still hit our cached result:

In [26]:
with pwf.tools._CacheTestExecutor() as exe:
    wf.slow_node_0.executor = exe
    out_shorter = wf.run(cfg, t_sleep=t_sleep * 42)
out_shorter.outputs

{'slept_for': OutputDataPort(value=1, annotation=None)}

In [27]:
# Clean up
shutil.rmtree(run_dir)

## Caching

Some atomic nodes may do extremely heavy lifting, e.g. invoking some external software. To avoid re-running expensive nodes at each invocation, we use `fleche` to cache results. Any fleche-decorated function will exploit a cache provided to its configuration at runtime. The `flowrep`/`pyiron_workflow` decorators are non-interacting with the `fleche` decorator, so if you use your node function as a plain function, `fleche` automatically does it's stuff:

In [28]:
import fleche

@pwf.atomic
@fleche.fleche
def cached_slow_node(t):
    time.sleep(t)
    return t

t0 = time.perf_counter()
cached_slow_node(0.5)
t1 = time.perf_counter()
cached_slow_node(0.5)
t2 = time.perf_counter()
print(f"First run took {t1-t0:.2f}s, second run (cache hit) took {t2-t1:.2f}s")

First run took 0.51s, second run (cache hit) took 0.00s


To exploit `fleche` during a run, you _must_ provide a cache at runtime.

We leave a full description of `fleche` and its different options and caching back-ends to the [fleche documentation](https://fleche.readthedocs.io/en/latest/). We just need to create a `fleche.caches.Cache`.

If you're using an out-of-process executor, you'll need to create some file-based cache. For our example here, we'll stay in the process and a simple memory cache will suffice.

In [29]:
from fleche.caches import Cache
from fleche.storage import ValueMemory, CallMemory

cache = Cache(
    values=ValueMemory({}),
    calls=CallMemory({}),
)

To leverage caching, all we do is pass this cache to the configuration at runtime:

In [30]:
n = cached_slow_node.pwf.node()
out1 = n.run(pwf.RunConfig(fleche_cache=cache), t=0.75)
out2 = n.run(pwf.RunConfig(fleche_cache=cache), t=0.75)
print(out1.duration.total_seconds(), out2.duration.total_seconds())
assert out1.outputs == out2.outputs

0.756709 0.000555


During a `run`, this is the _only_ way to leverage a fleche cache. Without explicitly providing a cache, caching during a run will be disabled -- even overriding fleche's usual `with` interface:

In [31]:
with fleche.cache(cache):
    out3 = n.run(t=0.33)
    out4 = n.run(t=0.33)
print(out3.duration.total_seconds(), out4.duration.total_seconds())

0.335612 0.335648


You don't need to worry about applying fleche's `wrap_executor` to your executors -- whenever you provide a `RunConfig.fleche_cache`, `pyiron_workflow` will take care of this wrapping for you.

Conclusion: `RunConfig.fleche_cache` filled with a cache -> cached with `fleche`; missing -> no caching.

## Validation

Beyond `flowrep`'s structural checks, `pyiron_workflow` can validate a recipe's python type hints across every edge. `validate(do_ontology=False)` returns a report whose `types` sub-report tells you whether the graph is type-`valid` and whether the check was `complete` (every edge had hints to compare).

For our completely unhinted `line` macro, validation is trivially affirmative:

In [32]:
report = pwf.node(line, "line").validate(do_ontology=False)
print("valid:   ", report.valid)
print("complete:", report.types.complete)
print(report.types.text)

valid:    True
complete: True
Type validation for 'line' (valid=True, complete=True): OK


But we can easily construct recipes which invalidate due to incompleteness (a typed input gets untyped source)

In [33]:
def expects_an_int(n: int) -> int:
    return n

@pwf.workflow
def unhinted_parent_input(x):
    crippling_expectations = expects_an_int(x)
    return crippling_expectations

unhinted_parent_input.pwf.validate()

/Users/liamhuber/dev/miniforge3/envs/pyiron12/lib/python3.12/site-packages/networkx/utils/backends.py:551: UserWarning: The hashes produced for directed graphs changed in version v3.5 due to a bugfix to track in and out edges separately (see documentation).
  return self.orig_func(*args, **kwargs)


Type validation for 'unhinted_parent_input' (valid=True, complete=False):
	unfulfilled edges:
		x->expects_an_int_0.n
Validation Report
Conforms: True

Or incorrectness (a typed input gets a _wrongly_ typed source)

In [34]:
def provides_a_float(x) -> float:
    return float(x)

@pwf.workflow
def bad_match(x):
    floaty = provides_a_float(x)
    grumpy = expects_an_int(floaty)
    return grumpy

bad_match.pwf.validate()

Type validation for 'bad_match' (valid=False, complete=True):
	invalid edges:
		provides_a_float_0.output_0->expects_an_int_0.n
Validation Report
Conforms: True

`validate` can _also_ check ontological (semantic) correctness via `semantikon`;
we save that demonstration for the discussion of `pyiron_workflow` with the broader pyiron ecosystem below.
As seen here, workflows without ontological annotations trivially pass this validation ("Conforms: True") just like un-typed workflows pass the typing validation.

# Managing recipes inside `pyiron_workflow`

Recipes and the nodes built from them are immutable: their topology is fixed. The `Workflow` object is the _one_ mutable thing in the WfMS -- it is the live editing
surface on which you grow a graph node-by-node and edge-by-edge, and from which you can emit a `flowrep` recipe when you are done.

Workflow objects give clearly named handles for graph mutations, all with inverse operations.

Adding IO:

In [35]:
wf = pwf.Workflow("wf")
wf.create_input("x", "m", "b")
wf.create_output("y")

[AddOutput(port=OutputPort(label='y', owner=<pyiron_workflow._wfms.workflow.Workflow object at 0x13fb9eff0>, type_hint=None, type_metadata=None))]

And removing it

In [36]:
wf.remove_input("b")
wf.remove_output("y")

[RemoveOutput(port=OutputPort(label='y', owner=<pyiron_workflow._wfms.workflow.Workflow object at 0x13fb9eff0>, type_hint=None, type_metadata=None))]

Adding nodes

In [37]:
wf.add_node(pwf.node(mul, "scale"))
wf.add_node(pwf.node(add, "shift"))

[AddNode(node=<pyiron_workflow._wfms.atomic.Atomic object at 0x148378e60>)]

And removing them

In [38]:
wf.remove_node("shift")

[RemoveNode(node=<pyiron_workflow._wfms.atomic.Atomic object at 0x148378e60>)]

Adding edges

In [39]:
source = fr.schemas.InputSource(port="m")
target = fr.schemas.TargetHandle(node="scale", port="x")
wf.add_edge(pwf.schemas.EdgeTuple(source, target))

wf.add_edge(
    pwf.schemas.EdgeTuple(
        fr.schemas.InputSource(port="x"),
        fr.schemas.TargetHandle(node="scale", port="y"),
    )
)

[AddEdge(edge=EdgeTuple(source=InputSource(node=None, port='x'), target=TargetHandle(node='scale', port='y')))]

And removing them

In [40]:
wf.remove_edge(pwf.schemas.EdgeTuple(source, target))

[RemoveEdge(edge=EdgeTuple(source=InputSource(node=None, port='m'), target=TargetHandle(node='scale', port='x')))]

There are also methods for renaming nodes and ports, and operations, some of which we'll talk about later.

Built on top of these handles, there are some pieces of syntactic sugar. For instance, we can add nodes by attribute assignment, create edges by connecting ports, and in one fell swoop we can create graph IO and wire it to child ports. Let's use these to redo some of the stuff we removed with our destructive operations above. We'll also exploit the fact that nodes with a _single output_ can be used in lieu of their output port.

In [41]:
wf.connect(wf.inputs.m, wf.scale.inputs.x)

wf.shift = pwf.node(add)
wf.connect(wf.scale, wf.shift.inputs.x)
wf.create_input_for(wf.shift.inputs.y, label="b")
wf.create_output_from(wf.shift, label="y")

[AddOutput(port=OutputPort(label='y', owner=<pyiron_workflow._wfms.workflow.Workflow object at 0x13fb9eff0>, type_hint=None, type_metadata=None)),
 AddEdge(edge=EdgeTuple(source=SourceHandle(node='shift', port='output_0'), target=OutputTarget(node=None, port='y')))]

In [42]:
print("nodes:", list(wf.nodes.keys()))
print("edges:", len(wf.edges))
# Children are reachable as attributes, too: `wf.scale`, `wf.shift`, ...
print("run:  ", wf.run(x=2, m=5, b=1).outputs["y"].value)

nodes: ['scale', 'shift']
edges: 5
run:   11


## Failing gracefully, and undo/redo

Each public mutation is an atomic, _undoable_ batch of operations, which we could see in the returned lists above. If anything inside a single call fails, the whole batch rolls back -- including parts that had already succeeded -- so the workflow is never left half-mutated.

Below, `add_edge` is asked to add two edges at once: a perfectly good one, and a type-incompatible one (`str -> int`). The good edge is applied first, then the bad edge fails type validation and the whole batch reverts: the edge count is unchanged.

In [43]:
@fr.atomic("x_str")
def give_str(x: str) -> str:
    return str(x)

@fr.atomic("x_int")
def need_int(x: int) -> int:
    return x

In [44]:
typed = pwf.Workflow("typed")
typed.create_input("x", type_hint=str)
typed.gs = pwf.node(give_str)
typed.ni = pwf.node(need_int)
typed.create_output_from(typed.ni, "x");

In [45]:
good = pwf.schemas.EdgeTuple(
    fr.schemas.InputSource(port="x"),
    fr.schemas.TargetHandle(node="gs", port="x"),
)
bad = pwf.schemas.EdgeTuple(
    fr.schemas.SourceHandle(node="gs", port="x_str"),
    fr.schemas.TargetHandle(node="ni", port="x"),  # str source into an int target
)

In [46]:
before = len(typed.edges)
try:
    typed.add_edge(good, bad)
except TypeError as error:
    print("rejected:", error)
print("edges unchanged:", len(typed.edges) == before, "(the good edge was rolled back along with the bad)")

rejected: Processing edge 'gs.x_str->ni.x' on 'typed', the type hint of the source (<class 'str'>) is not as-or-more-specific-than the target (<class 'int'>).
edges unchanged: True (the good edge was rolled back along with the bad)


Successful batches land on an undo stack. `undo()` reverses the last operation and `redo()` re-applies it, and each can take more than one batch at a time.

For instance, we can revisit our mx+b workflow:

In [47]:
wf = pwf.Workflow("wf")
wf.scale = pwf.node(mul)
wf.shift = pwf.node(add)
wf.connect(wf.scale, wf.shift.inputs.x)
wf.create_input_for(wf.scale.inputs.x, label="m")
wf.create_input_for(wf.scale.inputs.y, label="x")
wf.create_input_for(wf.shift.inputs.y, label="b")
wf.create_output_from(wf.shift, label="y")

print(f"{len(wf.nodes)} nodes, {len(wf.edges)} edges -- when complete")
wf.remove_node(wf.shift)
wf.remove_input("m")
print(f"{len(wf.nodes)} nodes, {len(wf.edges)} edges -- after removing some pieces")
wf.undo(2)
print(f"{len(wf.nodes)} nodes, {len(wf.edges)} edges -- ctrl+z!")
wf.redo()
print(f"{len(wf.nodes)} nodes, {len(wf.edges)} edges -- redo what's most recently undone: node removal")
wf.undo()
print(f"{len(wf.nodes)} nodes, {len(wf.edges)} edges -- back to where we started")

2 nodes, 5 edges -- when complete
1 nodes, 1 edges -- after removing some pieces
2 nodes, 5 edges -- ctrl+z!
1 nodes, 2 edges -- redo what's most recently undone: node removal
2 nodes, 5 edges -- back to where we started


### Dataclass Quality-of-Life

`flowrep` already allows you to take a python function returning a dataclass, and break the dataclass fields into output ports in the recipe. `pyiron_workflow` lets you go a step further and decorate dataclasses directly. Just like function decorations, these _stay as dataclasses_. Also like decorated functions, we extend the attributes available on them to include two new tool sets: one for making a node that transforms inputs into the dataclass instance, and another for the reverse direction: turning a dataclass instance into a set of separate output ports for each field.

Here is an "autoencoder" from dataclass to IO and back:

In [48]:
from pyiron_workflow._wfms import api as pwf

@pwf.dataclass(frozen=True)
class MyData:
    no_default: float
    foo: int = 42
    bar: str = "towel"
    not_a_field = 13

wf = pwf.Workflow("autoencoder")
wf.to_values = MyData.pwf_unpacking.node()
wf.to_dc = MyData.pwf.node()
wf.create_input_for(wf.to_values.inputs.dataclass)
wf.create_output_from(wf.to_dc)
for k in wf.to_values.outputs:
    wf.connect(wf.to_values.outputs[k], wf.to_dc.inputs[k])

autoencoder_result = wf.run(dataclass=MyData(no_default=1.1))
autoencoder_result.outputs["instance"]

OutputDataPort(value=MyData(no_default=1.1, foo=42, bar='towel'), annotation=<class '__main__.MyData'>)

In the middle of the workflow, the dataclass has been broken apart into its fields:

In [49]:
run.steps[0].outputs

{'output_0': OutputDataPort(value=10, annotation=None)}

## From workflow back to `flowrep` (and to python source)

At any point `wf.recipe` produces a `flowrep.schemas.WorkflowRecipe` -- the prospective,
declarative view of what you have built. `flowrep`'s compiler can render that recipe
straight back into runnable python source, closing the loop between visual/programmatic
editing and code:

In [50]:
rendered = fr.tools.flowrep2python(wf.recipe)
print(rendered.source)

from __future__ import annotations

import __main__
import flowrep
import typing

@flowrep.workflow("instance")
def compiled_from_workflow_recipe(dataclass):
    to_values_no_default, to_values_foo, to_values_bar = __main__.MyData._MyData_fields_to_outputs(dataclass)
    to_dc = __main__.MyData(no_default=to_values_no_default, foo=to_values_foo, bar=to_values_bar)
    return to_dc




## Operator injection

Arithmetic (and other) operators on ports and single-output nodes _inject_ the corresponding standard-library node into the graph automatically. Assigning a node-like value to a fresh attribute adds it under that label. So building "double the input" is a one-liner, and we've already seen that `create_output_from` mints a matching output port and
wires it in a single step:

In [51]:
inj = pwf.Workflow("inj")
inj.create_input("m")
inj.doubled = inj.inputs.m + inj.inputs.m  # operator injection adds a node
inj.create_output_from(inj.doubled, label="y")  # mint + wire an output in one call

print("nodes:", list(inj.nodes.keys()))
print("run:  ", inj.run(m=4).outputs["y"].value)

nodes: ['doubled']
run:   8


We can chain together these injections, and we always get back a single node that is ready to be added to its injecting scope, but it isn't parented yet and we don't _automatically_ mutate our injecting scope:

In [52]:
linear = pwf.Workflow("linear")
linear.create_input("m", "x", "b", type_hint=int)
y = linear.inputs.m * linear.inputs.x + linear.inputs.b

print(y.owner, len(linear.nodes))

None 0


We can't inject it somewhere else, because it knows it needs the sources it was injected on

In [53]:
try:
    inj.the_kid_is_not_my_son = y
except ValueError as e:
    print(e)

Port 'linear.inputs.m' is not owned by the inputs of context mutable graph 'inj', nor an output of that graph's child.


In [54]:
linear.y = y
linear.create_output_from(linear.y, label="y")
linear.run(m=2, x=3, b=0.5).outputs

{'y': OutputDataPort(value=6.5, annotation=None)}

Of course, if we create a free-standing injection, we can later add it anywhere

In [55]:
def identity(x):
    return x

cubic_term = pwf.node(identity) * pwf.node(identity) * pwf.node(identity)

linear.cubic = cubic_term
for inp in linear.cubic.inputs.values():
    linear.connect(linear.inputs.x, inp)
linear.create_output_from(linear.cubic, label="x3")

linear.run(m=2, x=3, b=0.5).outputs

{'y': OutputDataPort(value=6.5, annotation=None),
 'x3': OutputDataPort(value=27, annotation=None)}

## Grouping, locking, and their inverses

`group` collects sibling nodes into a nested sub-`Workflow` (still mutable).
`lock_subgraph` freezes a sub-workflow into an immutable `Macro` -- and once locked,
attempts to mutate its children raise a helpful error telling you exactly what to do.

As a playground to demonstrate this, let's rebuild our y=m*x + b example using the syntactic sugar we've seen so far

In [56]:
wf = pwf.Workflow("linear_sugar")
for inp in ("m", "x", "b"):
    wf

# Add nodes by attribute assignment
wf.scale = pwf.node(mul)
wf.shift = pwf.node(add)

# Create linked inputs
wf.create_input_for(wf.scale.inputs.x, label="m")
wf.create_input_for(wf.scale.inputs.y, label="x")
wf.create_input_for(wf.shift.inputs.y, label="b")

# Wire up children
wf.connect(wf.scale, wf.shift.inputs.x)
# wf.shift.inputs.x = wf.scale  # There's no sugar for port assignment

# Create linked output
wf.create_output_from(wf.shift, label="y")

wf.nodes

MutableNodeMap(linear_sugar):
	'scale': Atomic
	'shift': Atomic

In [57]:
wf.group("grp", wf.scale, wf.shift)  # nested Workflow -- still editable
wf.lock_subgraph("grp")  # -> Macro -- now immutable
print("locked as:", type(wf.grp).__name__)

wf.nodes

locked as: Macro


MutableNodeMap(linear_sugar):
	'grp': Macro

The workflow now just has a single node -- the group. The two nodes that were previously the children are now nested inside that group:

In [58]:
wf.grp.nodes

NodeMap(linear_sugar.grp):
	'scale': Atomic
	'shift': Atomic

Since this is a static macro, we can't go in and rearrange its interior structure:

In [59]:
try:
    # try to rewire a child of the locked macro
    wf.grp.scale.connect_input(x=wf.grp.inputs.shift__y)
except TypeError as error:
    print("blocked: ", error)

blocked:  'linear_sugar.grp.scale' does not have a mutable owner, and so its inputs cannot be modified.Try unlocking owner to a mutable graph first.


While the internal structure of the workflow has changed, we've rewired all the IO to the group such that it still interfaces with the world in the same way:

In [60]:
wf.run(m=2, x=3, b=0.4).outputs

{'y': OutputDataPort(value=6.4, annotation=None)}

`unlock_subgraph` is the inverse of `lock_subgraph` (`Macro` back to `Workflow`),
and `ungroup` is the inverse of `group` (flatten the children back up to the parent):

In [61]:
wf.unlock_subgraph("grp")
print("unlocked as:", type(wf.grp).__name__)
wf.ungroup("grp")

wf.nodes

unlocked as: Workflow


MutableNodeMap(linear_sugar):
	'grp_scale': Atomic
	'grp_shift': Atomic

This operation isn't idempotent on the names of the nodes in the workflow -- we've picked up the group name as a prefix -- but the operation is preserved

In [62]:
wf.run(m=2, x=3, b=0.3).outputs

{'y': OutputDataPort(value=6.3, annotation=None)}

# Ecosystem integration

`pyiron_workflow` is not the only element of the pyiron universe that interacts with `flowrep` recipes and data structures.
In particular, `semantikon` -- the pyiron package for knoweldge graphs and ontological analysis -- also digests them natively.
That means any workflow annotated ontologically by `semantikon` is immediately available for analysis by `semantikon`!

Here is a tiny ontology-annotated pipeline. Each node declares the semantic `type`
of its data and the properties it confers. `sell` carries OWL _restrictions_: it can
only act on clothes that are both `cleaned` and have a `color`.

In [63]:
import rdflib
import semantikon

EX = rdflib.Namespace("http://example.org/")
uri_cleaned = semantikon.SemantikonURI(EX.cleaned)
uri_color = semantikon.SemantikonURI(EX.color)


class Clothes:
    pass


@pwf.atomic
def dye(clothes: semantikon.u(Clothes, uri=EX.Clothes), color="blue") -> semantikon.u(
    Clothes,
    uri=EX.Clothes,
    triples=(EX.hasProperty, uri_color),
    derived_from="inputs.clothes",
):
    return clothes


@pwf.atomic
def wash(clothes: semantikon.u(Clothes, uri=EX.Clothes)) -> semantikon.u(
    Clothes,
    uri=EX.Clothes,
    triples=(EX.hasProperty, uri_cleaned),
    derived_from="inputs.clothes",
):
    return clothes


@pwf.atomic
def sell(
    clothes: semantikon.u(
        Clothes,
        uri=EX.Clothes,
        restrictions=(
            ((rdflib.OWL.onProperty, EX.hasProperty), (rdflib.OWL.someValuesFrom, EX.cleaned)),
            ((rdflib.OWL.onProperty, EX.hasProperty), (rdflib.OWL.someValuesFrom, EX.color)),
        ),
    ),
) -> int:
    return 10

@pwf.workflow
def storefront(clothes):
    dyed = dye(clothes)
    washed = wash(dyed)
    money = sell(washed)
    return money

This is a usual `pyiron_workflow` workflow which we can run

In [64]:
wf = storefront.pwf.node()
out = wf.run(clothes=Clothes())
out.outputs

{'money': OutputDataPort(value=10, annotation=None)}

But we can also get a knowledge graph right from its recipe

In [65]:
semantikon.get_knowledge_graph(wf.recipe)

<Graph identifier=Nffa8c4fa9b93453f8b867de2acdf5763 (<class 'rdflib.graph.Graph'>)>

or validate its ontological correctness

In [66]:
semantikon.validate_values(wf.recipe)

(True,
 <Graph identifier=Nb345924735f643989ba387ebc20e8718 (<class 'rdflib.graph.Graph'>)>,
 'Validation Report\nConforms: True\n')

We earlier saw that the `validate` methods on pwf objects had space for returning an ontological validation report.
We can use this shortcut to access information from `semantikon` directly.
Let's build an ontologically _failing_ workflow to see this in action:

In [67]:
@pwf.workflow
def hasty_storefront(clothes):
    dyed = dye(clothes)  # never washed -- the `cleaned` property is missing
    money = sell(dyed)
    return money

validation = hasty_storefront.pwf.validate()
validation

Type validation for 'hasty_storefront' (valid=True, complete=False):
	unfulfilled edges:
		clothes->dye_0.clothes
Validation Report
Conforms: False
Results (1):
Constraint Violation in QualifiedValueShapeConstraintComponent (http://www.w3.org/ns/shacl#QualifiedMinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ rdf:type sh:PropertyShape ; sh:path <http://example.org/hasProperty> ; sh:qualifiedMinCount Literal("1", datatype=xsd:integer) ; sh:qualifiedValueShape [ sh:class <http://example.org/cleaned> ] ]
	Focus Node: sns:d1d8faebb317947192656aa25d5cbe0c_hasty_storefront-dye_0-outputs-clothes_data
	Result Path: <http://example.org/hasProperty>
	Message: Focus node does not conform to shape MinCount 1: [ sh:class <http://example.org/cleaned> ]

Type validation is fine (although incomplete), but ontologically "invalid" verdict came from a real SHACL reasoner running over the RDF
encoding of the workflow.
The report carries both a human-readable explanation and
the underlying `rdflib` graph.

Watch for further connections to knowledge management in the future, e.g. to [`courier`](https://github.com/pyiron/courier).

Although not in the pyiron universe, strictly speaking, the compatiblity of `flowrep` with `python-workflow-definition` (PWD) also gives the ability to export `pyiron_workflow` workflows to PWD files (at least for graphs PWD supports -- non-nested, and without flow controls):

In [68]:
wf = pwf.Workflow("flattened_dag")
wf.create_input("m", "x", "b")
wf.y = wf.inputs.m * wf.inputs.x + wf.inputs.b
wf.create_output_from(wf.y, "y")
wf.flatten()

pwd_recipe = fr.tools.flowrep2pwd(wf.recipe, m=2, x=3, b=0.5)
print(f"As a {type(pwd_recipe).__name__}:\n")
pwd_recipe.model_dump(mode="json")

As a PythonWorkflowDefinitionWorkflow:



{'version': '0.0.1',
 'nodes': [{'id': 0, 'type': 'input', 'name': 'm', 'value': 2},
  {'id': 1, 'type': 'input', 'name': 'x', 'value': 3},
  {'id': 2, 'type': 'input', 'name': 'b', 'value': 0.5},
  {'id': 3, 'type': 'output', 'name': 'y'},
  {'id': 4, 'type': 'function', 'value': 'flowrep.std.add'},
  {'id': 5, 'type': 'function', 'value': 'flowrep.std.mul'}],
 'edges': [{'target': 4,
   'targetPort': 'a',
   'source': 5,
   'sourcePort': 'product'},
  {'target': 4, 'targetPort': 'b', 'source': 2, 'sourcePort': None},
  {'target': 5, 'targetPort': 'a', 'source': 0, 'sourcePort': None},
  {'target': 5, 'targetPort': 'b', 'source': 1, 'sourcePort': None},
  {'target': 3, 'targetPort': None, 'source': 4, 'sourcePort': 'added'}]}

And we can round-trip back to `pyiron_workflow` via `flowrep` tools. Unfortunately, there is one hiccup: the PWD representation has no place to hold that `_operator` functions demand _positional_ arguments only, so we need to re-inject this information to the recipe so our WfMS knows how to invoke the function

In [69]:
round_trip_recipe, terminal_inputs = fr.tools.pwd2flowrep(pwd_recipe)

# Re-inject signature information
for node_label, node_recipe in round_trip_recipe.nodes.items():
    node_recipe.reference.restricted_input_kinds = {
        "a": fr.schemas.RestrictedParamKind.POSITIONAL_ONLY,
        "b": fr.schemas.RestrictedParamKind.POSITIONAL_ONLY,
    }

round_trip_wf = pwf.node(round_trip_recipe)
round_trip_wf.run(**terminal_inputs).outputs

{'y': OutputDataPort(value=6.5, annotation=None)}

# Compatibility and development plan

The legacy implementation of `pyiron_workflow` predates `flowrep` entirely, and differs from the WfMS discussed above in some critical ways:

- Legacy decorators transform functions in _node factories_ -- calling such a function produces a _node instance_ and not the naively expected function call result
- The legacy workflow object _holds execution state data_ -- i.e. the prospective recipe and retrospective data views are blended into a single object
  - Invoking a run without explicitly providing input data was possible, because the `pyiron_workflow` object itself held input data as state
  - The post-run state of the `pyiron_workflow` object was the real result, because all output data was also held as state, so `.run` outputs were a simple dictionary-like representation of terminal outputs
- In the legacy paradigm, execution flow was explicitly accessible and explicitly manageable, including syntactic sugar for designating "starting nodes" and execution order
- Legacy nodes could be connected and run as a graph _outside_ the context of a shared owner/parent graph objects, because the ground truth for edges/connections _lived on the IO ports themselves_

There are many small technical differences, but these four points represent the main philosophical differences between the two attacks, and explain most of the difference in features/behaviours.

Despite these differences, the old and new approaches have many similarities: named IO ports, breaking graphs into indivisible "atomic"/"function" objects vs objects with their own internal graph structure, nesting graphs and having graphs negotiate IO for their children, type checking new edges, node injection syntactic sugar for modifying workflows, etc.

Because of these many similarities, we are able to offer a basic level of compatiblity between _new_ `pyiron_workflow` nodes, and legacy `pyiron_workflow` nodes.
In particular, we offer a compatibility module with `@as_function_node` and `@as_macro_node` decorators which can be imported in lieu of the legacy decorators, which interpret legacy-formatted functions, but return new-style node objects.

For "atomic"/"function" nodes, this is very straightforward since they are both just plain python functions. Any legacy code decorated with `@as_function_node` where the decorator itself receives no arguments or only `*output_labels: str` output labeling directions can be used directly. Just like the legacy decorators and unlike the new `@atomic` decorator, this transforms the function into a (new-style!) _node factory_:

In [70]:
@pwf.compatibility.as_function_node
def legacy_add(x, y):
    return x + y

@pwf.compatibility.as_function_node
def legacy_mul(x, y):
    return x * y

The differences between `@as_macro_node` and `@workflow` are more pronounced, but any `@as_macro_node`-decorated function can be swapped for the new compatibility decorator as long as it takes no decorator arguments other than output labels, and as long as the function body is a straightforward construction of assigning new nodes as children and returning them and their outputs -- i.e. no flow control declarations, and no explicit execution flow management. The last critical step is that all the node-creating calls inside the function definition need to themselves be decorated with `@as_macro_node` or `@as_function_node`

In [71]:
@pwf.compatibility.as_macro_node
def legacy_line(self, x, m, b):
    self.scaled = legacy_mul(x, m)
    self.shifted = legacy_add(self.scaled, b)
    return self.shifted

The biggest thing to remember is that any code _using_ these new compatibility node factories is getting back out _new-style_ nodes. They will operate and run accordingly:

In [72]:
wf = legacy_line()
print(type(wf).__name__)

Macro


In [73]:
wf.run(m=10, x=2, b=0.3).outputs

{'shifted': OutputDataPort(value=20.3, annotation=None)}

This compatiblity layer is intended to ease the transition from the legacy custom `pyiron_workflow` parser for workflows-as-python onto the `flowrep`-based approach. For most of the common macros above, the main change is removing the `self` argument, and you are likely to have good luck getting an LLM to manage the conversion by providing one or two examples.

You can also generate `flowrep`-compliant python code directly from the resulting node objects using `flowrep`'s compiler functionality. That won't let you compile a workflow that already _has_ an underlying python reference function (which, deep under the hood, these legacy objects do), so we additionally slap on a convenience method that takes care of purging the reference prior to conversion for you:

In [74]:
rendered = legacy_line.flowrep_rendered()
print(rendered.source)

from __future__ import annotations

import __main__
import flowrep
import typing

@flowrep.workflow("shifted")
def legacy_line(x, m, b):
    legacy_mul = __main__.legacy_mul.decorated(x=x, y=m)
    legacy_add = __main__.legacy_add.decorated(x=legacy_mul, y=b)
    return legacy_add




However, given the similarity of the old and new decorated python declarations, it's probably easier for you (or your LLM) to simply change `@as_function_node` to `@atomic`, and purge the `self` terms from each `@as_macro_node` and swap it to `@workflow`. The most-likely problem to encounter on this route is the application of constants inside a legacy definition, e.g.

```python
@as_macro_node
def my_old_thing(x):
    self.child = some_child_call(x, 42)
    return self.child.outputs.some_channel
```

Since the legacy node objects _themselves stored data state_, this was doable.
With clean separation of recipes and data, it now becomes more difficult.
We're exploring the possibility of injecting constant-return nodes to accommodate this, so stay tuned.

## The plan

1. (We are here) new `flowrep`-based WfMS exists in a private submodule of `pyiron_workflow`, including a compatibility unit to generate new-style nodes from legacy `@as_function_node`- and `@as_macro_node`-decorated python code.
2. The new WfMS is promoted to the root of `pyiron_workflow`, while the legacy WfMS is demoted to a submodule.
3. The legacy WfMS and compatibility unit are removed.